In [ ]:
# ===================================
# ESSENTIAL LIBRARIES
# ===================================
import os
import shutil
import random
import warnings
import logging
from datetime import datetime

import numpy as np  
import pandas as pd  
import tqdm  

# ===================================
# VISUALIZATION
# ===================================
import matplotlib.pyplot as plt  
import seaborn as sn  
from matplotlib.colors import ListedColormap
import scikitplot as skplt  # Confusion matrix plot

# ===================================
# MACHINE LEARNING & METRICS
# ===================================
from sklearn.utils import shuffle
from sklearn.model_selection import (
    train_test_split,       # Hold-out
    LeaveOneOut,            # LOOCV
    StratifiedKFold         # K-fold
)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    SelectKBest,
    mutual_info_classif,
    SequentialFeatureSelector
)
from sklearn.ensemble import RandomForestClassifier  # Also for SFS
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    roc_curve, 
    auc, 
    precision_recall_curve,
    confusion_matrix, 
    classification_report,
    ConfusionMatrixDisplay
)

# ===================================
# FEATURE SELECTION
# ===================================
import mrmr
from mrmr import mrmr_classif  # Minimum Redundancy Maximum Relevance (mRMR)

# ===================================
# CLASSIFIERS
# ===================================
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier

# Preparation of the Classification Dataset

Before we can perform classification, we need to properly organize the features extracted with PyRadiomics. This script takes the CSV output file from PyRadiomics and processes it to separate the actual features from the metadata. These prepared features can then be used for various classification tasks, such as:
- Distinguishing in situ vs. invasive tumors
- Predicting triple-negative breast cancers
- Or any other clinical label of interest

1. **Load the feature file**:  
   The PyRadiomics output (a `.csv` file) is loaded as a Pandas DataFrame.

In [ ]:
data = pd.read_csv('../utilities/pyradiomics_features.csv')

2. **Split metadata and features:**
   
   The first 39 columns of the CSV usually contain metadata (such as image and mask paths, PyRadiomics version, etc.). We separate this into:
   - info: contains metadata
   - feature: contains the radiomics features only

In [ ]:
info = data.drop(columns=data.columns[39:])
feature = data.drop(columns=data.columns[:39])

3. **Extract patient IDs**:
   
   The patient identifier can be extracted from the Mask column, which contains the full path to the segmentation files.

   To extract the patient ID from the Mask column, we remove the absolute path and file extension.
   This is necessary because PyRadiomics includes full file paths in its output. We isolate only the patient folder name (e.g., DUKE_001), which will be used as a unique identifier in the feature table.

    ⚠️ Make sure to adapt the path in the str.replace() function to match your current dataset location.
    This code supports both .nii.gz and .nii extensions.

In [ ]:
print(data['Mask'].loc[0])

In [ ]:
patient_id = pd.DataFrame()
patient_id['ID'] = data['Mask']

patient_id['ID'] = patient_id['ID'].str.replace(
    'C:\\Users\\rocco\\Documents\\MaterialeTesiMagistrale\\segmentations\\expert\\',  # ⚠️ Update if needed
    '', regex=False
).str.replace('.nii.gz', '', regex=False).str.replace('.nii', '', regex=False)

4. **Insert patient ID into the feature set**:

   We add the extracted patient ID as the first column of the feature DataFrame, which will later be used as an index or identifier for classification.

In [ ]:
feature.insert(0, 'ID', patient_id['ID'])

5. **Save the dataframe as CSV**:

In [ ]:
feature.to_csv('feature.csv', index=False)
print("\nCSV file 'feature.csv' saved.")

In [ ]:
label = pd.DataFrame()
label['ID'] = feature['ID']
label['labels'] = [0,1,0]
label.to_csv('label.csv', index=False)

# Training, validation and test classification

6. **Preparing the feature dataframe and labels array**:

   - Load the feature.csv and label.csv files containing the radiomic features and the corresponding labels.
   - Sort both DataFrames by patient ID to ensure correct alignment.
   - Verify that the IDs match exactly between the two DataFrames; otherwise, raise an error.
   - Drop the ID column from both datasets to keep only the numeric data.
   - Convert the label column into a one-dimensional NumPy array, ready to be used in classification models.

In [ ]:
import pandas as pd
import numpy as np

feature = pd.read_csv('./feature.csv').sort_values(by='ID').reset_index(drop=True)
label = pd.read_csv('./label.csv').sort_values(by='ID').reset_index(drop=True)

if feature['ID'].tolist() != label['ID'].tolist():
    raise ValueError("DataFrames are not aligned correctly!")

label = label.drop(columns=['ID']).squeeze()
feature = feature.drop(columns=['ID'])

if isinstance(label, pd.DataFrame) and label.shape[1] == 1:
    label = label.iloc[:, 0].values
elif isinstance(label, pd.Series):
    label = label.values

assert label.ndim == 1

7. **Define Utility Functionsn**:
   Before running the classification tasks, make sure to define the following utility functions inside the notebook kernel:

    - ROC Curve Plotting Function  

    - Folder Creation Function

In [ ]:
def ROC_curve(true_label, probs, run, fea_num, output_dir):
    """
    Plot and save the ROC curve for a classification model, and return performance metrics.

    This function is designed to be reusable across different models, assuming that the following
    variable names are consistently used throughout the pipeline:

    Required variable names for correct usage:
    - train_probs: Probabilities predicted on the training set
    - train_pred: Predictions on the training set
    - test_probs: Probabilities predicted on the test set
    - test_pred: Predictions on the test set
    - label_train: Ground truth labels for the training set
    - label_test: Ground truth labels for the test set
    - feature_train: Feature matrix used for training
    - feature_test: Feature matrix used for testing

    Parameters:
        true_label (array-like): Ground truth binary labels for the test set.
        probs (array-like): Predicted probabilities for the positive class.
        run (int or str): Identifier for the current run (used in the plot title and filename).
        fea_num (int or str): Number of features used (for filename annotation).
        output_dir (str): Directory where the ROC plot image will be saved.

    Returns:
        fpr (ndarray): False Positive Rates.
        tpr (ndarray): True Positive Rates.
        thresholds (ndarray): Thresholds corresponding to each FPR/TPR pair.
        roc_auc (float): Area Under the ROC Curve (AUC).
    """
    
    fpr, tpr, thresholds = roc_curve(true_label, probs)
    roc_auc = auc(fpr, tpr)
    
    # Styling the plot
    plt.figure(figsize=(7, 7))
    plt.plot(fpr, tpr, color='#007acc', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='black', linestyle='--', lw=1)  # Reference line
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.02])
    plt.xlabel('False Positive Rate (FP / FP + TN)', fontsize=12)
    plt.ylabel('True Positive Rate (TP / TP + FN)', fontsize=12)
    plt.title(f'ROC Curve - Run {run}, Features {fea_num}', fontsize=14)
    
    plt.legend(loc="lower right", fontsize=10)
    plt.tick_params(axis='both', which='major', labelsize=10)
    plt.tight_layout()
    
    # Save plot
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, f'roc_run{run}_fea{fea_num}_auc_{roc_auc:.2f}.png')
    print(f"Saving ROC curve as: {filename}")
    plt.savefig(filename, dpi=300)
    plt.close()
    
    return fpr, tpr, thresholds, roc_auc

In [ ]:
def create_unique_run_dir(base_dir, model_name):
    """
    Function to create a uniquely named output directory for saving ROC curves of a model run
    
    Creates a timestamped subdirectory inside the specified base directory to store 
    model-specific outputs (e.g., ROC curve images) for a given run.

    Parameters:
        base_dir (str): Path to the root output directory.
        model_name (str): Name of the model (used in the folder naming).

    Returns:
        str: Full path to the newly created, unique output directory.
    """
    os.makedirs(base_dir, exist_ok=True)  # Ensure base directory exists
    timestamp = datetime.now().strftime("%H-%M-%S")  # Current time for uniqueness
    new_run_dir = os.path.join(base_dir, f"{timestamp}_{model_name}_LOPO_ROC_Curves")
    os.makedirs(new_run_dir, exist_ok=True)  # Create the new run directory
    return new_run_dir


## Leave-One-Out Cross Validation

### Description of the LOOCV Classification (1)

This script is designed to perform a classification analysis using machine learning techniques in a Leave-One-Patient-Out (LOPO) context. Model evaluation occurs through several stages, including model selection, feature selection, training and validation, calculation of performance metrics, and the generation of the ROC curve.

1. **Model Selection**: The user can choose from four machine learning models: 
   - XGBoost
   - K-Nearest Neighbors (KNN)
   - Support Vector Machine (SVM)
   - Random Forest
   
   Based on the selected model, specific parameters are set for each algorithm.

2. **Feature Selection**: The user can choose one of two feature selection methods:
   - **MRMR** (Minimum Redundancy Maximum Relevance): Selects a specified number of features based on their relevance and redundancy.
   - **Mutual Information**: Selects the best features based on mutual information between the features and the target class.
   
    The number of features to select is provided by the user, and progressive sets of features are created, gradually increasing the number of selected features. This process is applied to the entire dataset, not just the training data, ensuring that the feature selection reflects the full dataset's characteristics.


4. **LOPO Cross-Validation (Leave-One-Patient-Out)**: For each run, the LOPO technique is used, where each patient is used as the test set in a single iteration, while the remaining patients are used for training. This approach ensures that the model is tested on unseen cases during training.

5. **Feature Standardization**: The selected features are standardized (zero mean and unit variance) separately for each fold of cross-validation, to prevent data leakage between training and test data.

6. **Training and Prediction**: The selected model is trained on the training data and tested on the test data for each fold. Predictions are collected to calculate performance metrics.

7. **Evaluation Metrics Calculation**: For each run, the following performance metrics are calculated:
   - **Accuracy**: The percentage of correct predictions.
   - **Precision**: The percentage of correctly classified positive samples among all predicted positives.
   - **Recall**: The percentage of correctly classified positive samples among all true positives.
   - **F1-score**: The harmonic mean of precision and recall.
   - **Confusion Matrix**: Represents the number of true positives, true negatives, false positives, and false negatives.

8. **ROC Curve and AUC**: For each run, the ROC (Receiver Operating Characteristic) curve is generated, which plots the trade-off between the false positive rate (FPR) and the true positive rate (TPR) for all classification thresholds. The AUC (Area Under the Curve) is also calculated, providing an overall measure of the model's performance.

9. **Saving Results**: Results, including model parameters, evaluation metrics for each run, and the ROC curve, are saved in CSV files and image files:
   - **metrics.csv**: Contains performance metrics for each run, threshold, and feature set. This is saved in the output folder specified for each run.
   - **auc_mean_for_run.csv**: Contains AUCs for each run with statistics such as mean, variance, and standard deviation. This file is also saved in the same output folder.
   - **ROC Curve**: A graph of the ROC curve is saved as a PNG image in a unique folder for each run, including the timestamp and the model name. The destination folder is automatically created for each run with a unique name that includes the model, feature selection method, and timestamp. 

   The main folder where all files are saved is a subfolder of `./data/00-Original/`, specifically created for the selected model and feature selection method, such as `./data/00-Original/XGBOOST_MRMR_LOPO/` for an analysis with the XGBoost model and MRMR method. Inside this folder, additional subfolders are created for each run, ensuring that results are saved in an organized manner.

The code approach allows for multiple iterations with different combinations of selected features, providing a thorough evaluation of model performance for each configuration. Results are automatically saved in a unique directory for each run, facilitating subsequent analysis.


In [ ]:
# Logging
logging.basicConfig(level=logging.INFO)

# --- Model selection ---
valid_models = {"xgboost", "knn", "svm", "random_forest", "naive_bayes", "decision_tree", "logistic", "mlp"}
model = input("Choose the model (xgboost, knn, svm, random_forest, naive_bayes, decision_tree, logistic, mlp): ").strip().lower()
while model not in valid_models:
    model = input("Invalid choice. Enter a valid model name: ").strip().lower()

# --- Feature selection method ---
feature_methods = {'mrmr', 'mutualinfo'}
feature_selection = input("Choose feature selection method (mrmr, mutualinfo): ").strip().lower()
while feature_selection not in feature_methods:
    feature_selection = input("Invalid choice. Enter a valid method name: ").strip().lower()

# --- Output directory ---
def create_unique_run_dir(base_dir, model_name):
    os.makedirs(base_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%H-%M-%S")
    new_run_dir = os.path.join(base_dir, f"{timestamp}_{model_name}_LOPO_ROC_Curves")
    os.makedirs(new_run_dir, exist_ok=True)
    return new_run_dir

base_output_dir = "./data/00-Original/"
model_dir = os.path.join(base_output_dir, f"{model.upper()}_{feature_selection}_LOPO")
output_dir = create_unique_run_dir(model_dir, model.upper())
logging.info(f"Results will be saved in: {output_dir}")

# --- Number of runs ---
num_runs = int(input("Enter the number of runs: "))
runs = list(range(1, num_runs + 1))

# --- Feature selection ---
nums_total_fea = int(input("Enter number of features to select: "))
if feature_selection == 'mrmr':
    all_selected_features = mrmr_classif(feature, label, K=nums_total_fea)
    progressive_feature_sets = [all_selected_features[:i] for i in range(1, nums_total_fea + 1)]
else:
    selector = SelectKBest(score_func=mutual_info_classif, k=nums_total_fea)
    selector.fit(feature, label)
    selected_feature_names = feature.columns[selector.get_support()]
    progressive_feature_sets = [selected_feature_names[:i] for i in range(1, nums_total_fea + 1)]

# --- Model parameters ---
model_params_filename = os.path.join(output_dir, f"{model.upper()}_params.txt")
model_params = {}

if model == "xgboost":
    model_params = {
        'objective': 'binary:logistic', 'max_depth': 6, 'learning_rate': 0.01,
        'n_estimators': 300, 'lambda': 0.5, 'alpha': 0.5, 'eval_metric': 'auc',
        'scale_pos_weight': 25/46
    }
    
elif model == "svm":
    model_params = {
        'C': 0.5, 'kernel': 'rbf', 'gamma': 'scale', 'shrinking': True,
        'probability': True, 'class_weight': 'balanced', 'random_state': 42
    }
    
elif model == "knn":
    model_params = {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'cosine'}
    
elif model == "random_forest":
    model_params = {
        'n_estimators': 200, 'criterion': 'gini', 'max_depth': 7,
        'min_samples_split': 5, 'min_samples_leaf': 2, 'random_state': 42
    }

elif model == "naive_bayes":
    model_params = {} 

elif model == "decision_tree":
    model_params = {'criterion': 'gini', 'max_depth': 8,'random_state': 42}

elif model == "logistic":
    model_params = {'C': 0.1, 'solver': 'saga', 'random_state': 42}
    
elif model == "mlp":
    model_params = {'hidden_layer_sizes': (64, 32), 'activation': 'tanh', 'solver': 'lbfgs','alpha': 0.01, 'max_iter': 1000, 'random_state': 42}
    
with open(model_params_filename, 'w') as f:
    f.write(f"Model: {model.upper()}\n\n")
    for param, value in model_params.items():
        f.write(f"{param}: {value}\n")
    f.write("\nTraining and Evaluation will start...\n")

# --- Standardization and LOPO ---
scaler = StandardScaler()
all_metrics = []

for run in runs:
    for fea_num, selected_features in enumerate(progressive_feature_sets, 1):
        feature_selected = feature[selected_features]
        feature_selected = pd.DataFrame(feature_selected)

        all_preds, all_probs, all_true_labels = [], [], []
        kf = LeaveOneOut()

        model_run_filename = os.path.join(output_dir, f"{model.upper()}_params_run{run}_features.txt")
        with open(model_run_filename, 'a') as f:
            f.write(f"\nRun {run}, {fea_num} Features:\n")
            for feature_name in selected_features:
                f.write(f"  - {feature_name}\n")

        for train_idx, test_idx in kf.split(feature_selected, label):
            X_train, X_test = feature_selected.iloc[train_idx], feature_selected.iloc[test_idx]
            y_train, y_test = label[train_idx], label[test_idx]

            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            if model == "xgboost":
                clf = XGBClassifier(**model_params)
            elif model == "svm":
                clf = SVC(**model_params)
            elif model == "knn":
                clf = KNeighborsClassifier(**model_params)
            elif model == "random_forest":
                clf = RandomForestClassifier(**model_params)

            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)
            probs = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else preds

            all_preds.extend(preds)
            all_probs.extend(probs)
            all_true_labels.extend(y_test)

        fpr, tpr, thresholds, roc_auc = ROC_curve(all_true_labels, all_probs, run, fea_num, output_dir)

        for threshold in thresholds:
            preds = (np.array(all_probs) >= threshold).astype(int)
            conf_matrix = confusion_matrix(all_true_labels, preds)
            all_metrics.append({
                "run": run, "auc": roc_auc, "features": fea_num, "threshold": threshold,
                "accuracy": accuracy_score(all_true_labels, preds),
                "precision": precision_score(all_true_labels, preds, zero_division=0),
                "recall": recall_score(all_true_labels, preds, zero_division=0),
                "f1": f1_score(all_true_labels, preds),
                "confusion_matrix": conf_matrix
            })

# --- Save metrics ---
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(os.path.join(output_dir, "metrics.csv"), index=False)

auc_values = metrics_df.groupby('run')['auc'].apply(list)
auc_mean_for_run = pd.DataFrame({
    "run": auc_values.index,
    "AUC": auc_values.tolist(),
    "mean": auc_values.apply(np.mean),
    "variance": auc_values.apply(np.var),
    "std_dev": auc_values.apply(np.std)
})
auc_mean_for_run.to_csv(os.path.join(output_dir, "auc_mean_for_run.csv"), index=False)




# =================================================
# OPTIONAL: SELECT BEST CASE CLOSEST TO AUC MEAN
# =================================================

best_case_option = input("Do you want to select the best case (closest to AUC mean) and clean up? (yes/no): ").strip().lower()

if best_case_option == 'yes':
    # Compute overall mean AUC
    overall_mean_auc = auc_mean_for_run["mean"].mean()
    metrics_df['auc_distance'] = abs(metrics_df['auc'] - overall_mean_auc)
    best_entry = metrics_df.loc[metrics_df['auc_distance'].idxmin()]

    best_run = best_entry["run"]
    best_fea_num = best_entry["features"]
    best_auc = best_entry["auc"]

    # Create directory to store best ROC images
    best_output_dir = os.path.join(output_dir, "best_ROC_images")
    os.makedirs(best_output_dir, exist_ok=True)

    # Copy the selected best ROC image
    best_image_filename = os.path.join(output_dir, f'roc_run{best_run}_fea{best_fea_num}_auc_{best_auc:.2f}.png')
    best_image_copy_path = os.path.join(best_output_dir, f'roc_run{best_run}_fea{best_fea_num}_auc_{best_auc:.2f}.png')
    if os.path.exists(best_image_filename):
        shutil.copy(best_image_filename, best_image_copy_path)
    else:
        logging.warning(f"ROC image {best_image_filename} not found.")

    # Save ROC metrics of the best case
    best_roc_df = metrics_df[(metrics_df["run"] == best_run) & (metrics_df["features"] == best_fea_num)]
    best_roc_df.to_csv(os.path.join(best_output_dir, "best_roc_metrics.csv"), index=False)

    # Clean up output directory, preserving only key files
    for file_name in os.listdir(output_dir):
        file_path = os.path.join(output_dir, file_name)
        if (
            os.path.isdir(file_path) or
            file_name.startswith("auc_mean_for_run") or
            file_name.startswith("metrics") or
            "best_ROC_images" in file_path
        ):
            continue
        try:
            os.remove(file_path)
            logging.info(f"Deleted: {file_path}")
        except Exception as e:
            logging.warning(f"Could not delete {file_path}: {e}")

    logging.info("Cleanup complete. Best ROC images preserved.")

else:
    logging.info("Skipped best case selection and cleanup.")

In [ ]:
feature = feature.drop(columns=['laterality'])